# Polymarket Notebook

This project fetches and analyzes public Polymarket market data.


## 1. Import Libraries and Configure API Access


In [1]:
import ast
import json
import time
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

API_BASES = [
    "https://gamma-api.polymarket.com",
]

DEFAULT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "polymarket-analysis-notebook/1.0",
}

REQUEST_TIMEOUT = 20


## 2. Fetch Polymarket API Data (Open Events)


In [2]:
def request_json(
    path: str,
    params: Optional[Dict[str, Any]] = None,
    max_retries: int = 3,
    backoff_seconds: float = 1.25,
) -> Tuple[Any, str]:
    params = params or {}
    last_error: Optional[Exception] = None

    for base in API_BASES:
        url = f"{base}{path}"
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.get(
                    url,
                    params=params,
                    headers=DEFAULT_HEADERS,
                    timeout=REQUEST_TIMEOUT,
                )
                response.raise_for_status()
                return response.json(), url
            except Exception as exc:  # noqa: BLE001
                last_error = exc
                if attempt < max_retries:
                    time.sleep(backoff_seconds * attempt)

    raise RuntimeError(f"Failed to fetch {path} from all base URLs") from last_error


def fetch_paginated(
    path: str,
    page_size: int = 100,
    max_pages: int | None = None,
    extra_params: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    all_records: List[Dict[str, Any]] = []

    page = 0
    while True:
        offset = page * page_size
        params = {"limit": page_size, "offset": offset}
        if extra_params:
            params.update(extra_params)

        data, used_url = request_json(path, params=params)

        if isinstance(data, dict):
            records = (
                data.get("data")
                or data.get("markets")
                or data.get("events")
                or data.get("items")
                or []
            )
        elif isinstance(data, list):
            records = data
        else:
            records = []

        if not records:
            print(f"No more records at offset={offset} from {used_url}")
            break

        all_records.extend(records)

        if len(records) < page_size:
            break

        page += 1
        if max_pages is not None and page >= max_pages:
            print(f"Reached configured max_pages={max_pages} for {path}")
            break

    return pd.DataFrame(all_records)


def _to_list(value: Any) -> list:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        for loader in (json.loads, ast.literal_eval):
            try:
                out = loader(s)
                if isinstance(out, list):
                    return out
            except Exception:
                pass
    return []


def _derive_event_category(row: pd.Series) -> str:
    tags = _to_list(row.get("tags"))
    for t in tags:
        if isinstance(t, dict):
            for k in ("label", "name", "category", "slug"):
                v = t.get(k)
                if isinstance(v, str) and v.strip():
                    return v.strip()
        elif isinstance(t, str) and t.strip():
            return t.strip()

    series = row.get("series")
    if isinstance(series, dict):
        for k in ("name", "slug", "title"):
            v = series.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()

    for k in ("seriesSlug", "series"):
        v = row.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()

    return "uncategorized"


MARKETS_PAGE_SIZE = 100
MARKETS_MAX_PAGES = 20
EVENTS_PAGE_SIZE = 100
EVENTS_MAX_PAGES = None
EVENTS_QUERY_PARAMS = {"closed": "false", "active": "true"}

markets_raw = fetch_paginated("/markets", page_size=MARKETS_PAGE_SIZE, max_pages=MARKETS_MAX_PAGES)
events_raw = fetch_paginated(
    "/events",
    page_size=EVENTS_PAGE_SIZE,
    max_pages=EVENTS_MAX_PAGES,
    extra_params=EVENTS_QUERY_PARAMS,
)

# Safety open filter in case endpoint params are partially ignored.
if not events_raw.empty:
    if "closed" in events_raw.columns:
        events_raw = events_raw[events_raw["closed"].fillna(False) == False]
    if "active" in events_raw.columns:
        events_raw = events_raw[events_raw["active"].fillna(False) == True]

# Derive a usable category from tags/series immediately.
if not events_raw.empty:
    events_raw = events_raw.copy()
    events_raw["derived_category"] = events_raw.apply(_derive_event_category, axis=1)

print(f"Markets fetched: {len(markets_raw):,} (cap={MARKETS_PAGE_SIZE * MARKETS_MAX_PAGES:,})")
events_cap_label = "unbounded" if EVENTS_MAX_PAGES is None else f"{EVENTS_PAGE_SIZE * EVENTS_MAX_PAGES:,}"
print(f"Open events fetched: {len(events_raw):,} (cap={events_cap_label})")


Reached configured max_pages=20 for /markets
Markets fetched: 2,000 (cap=2,000)
Open events fetched: 10,778 (cap=unbounded)


## 3. Inspect Response Structure and Fields


In [3]:
print("Markets columns:", sorted(markets_raw.columns.tolist())[:30], "...")
print("Events columns: ", sorted(events_raw.columns.tolist())[:30], "...")

if not markets_raw.empty:
    display(markets_raw.head(3))

if not events_raw.empty:
    display(events_raw.head(3))

if not events_raw.empty and "derived_category" in events_raw.columns:
    print("Top derived categories in open events:")
    display(events_raw["derived_category"].value_counts(dropna=False).head(15).to_frame("count"))


Markets columns: ['acceptingOrders', 'acceptingOrdersTimestamp', 'active', 'approved', 'archived', 'automaticallyActive', 'bestAsk', 'bestBid', 'clearBookOnStart', 'clobRewards', 'clobTokenIds', 'closed', 'competitive', 'conditionId', 'createdAt', 'customLiveness', 'cyom', 'deploying', 'deployingTimestamp', 'description', 'enableOrderBook', 'endDate', 'endDateIso', 'eventStartTime', 'events', 'featured', 'feeType', 'feesEnabled', 'funded', 'gameStartTime'] ...
Events columns:  ['active', 'archived', 'automaticallyActive', 'cantEstimate', 'closed', 'color', 'commentCount', 'competitive', 'countryName', 'createdAt', 'createdBy', 'creationDate', 'cumulativeMarkets', 'cyom', 'deploying', 'deployingTimestamp', 'derived_category', 'description', 'elapsed', 'electionType', 'enableNegRisk', 'enableOrderBook', 'endDate', 'ended', 'estimateValue', 'estimatedValue', 'eventCreators', 'eventDate', 'eventMetadata', 'eventWeek'] ...


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,description,outcomes,outcomePrices,volume,active,closed,marketMakerAddress,createdAt,updatedAt,new,featured,submitted_by,archived,resolvedBy,restricted,groupItemTitle,groupItemThreshold,questionID,enableOrderBook,orderPriceMinTickSize,orderMinSize,volumeNum,liquidityNum,endDateIso,startDateIso,hasReviewedDates,volume24hr,volume1wk,volume1mo,volume1yr,clobTokenIds,umaBond,umaReward,volume24hrClob,volume1wkClob,volume1moClob,volume1yrClob,volumeClob,liquidityClob,customLiveness,acceptingOrders,negRisk,negRiskRequestID,events,ready,funded,acceptingOrdersTimestamp,cyom,competitive,pagerDutyNotificationEnabled,approved,clobRewards,rewardsMinSize,rewardsMaxSpread,spread,oneWeekPriceChange,oneMonthPriceChange,lastTradePrice,bestBid,bestAsk,automaticallyActive,clearBookOnStart,seriesColor,showGmpSeries,showGmpOutcome,manualActivation,negRiskOther,umaResolutionStatuses,pendingDeployment,deploying,deployingTimestamp,rfqEnabled,holdingRewardsEnabled,feesEnabled,requiresTranslation,feeType,oneDayPriceChange,oneHourPriceChange,negRiskMarketID,umaResolutionStatus,gameStartTime,eventStartTime,groupItemRange
0,540816,Russia-Ukraine Ceasefire before GTA VI?,0x9c1a953fe92c8357f1b646ba25d983aa83e90c525992...,russia-ukraine-ceasefire-before-gta-vi-554,,2026-07-31T12:00:00Z,74282.866,2025-05-02T15:48:00.174Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if if there...","[""Yes"", ""No""]","[""0.545"", ""0.455""]",1499768.430793972,True,False,,2025-05-02T15:03:10.397014Z,2026-04-13T18:48:16.690283Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x6A9D222616C90FcA5754cd1333cFD9b7fb6a4F74,True,Russia-Ukraine Ceasefire,1,0x74dcd73f29877722e217723e10f2e8f9a888976f7cfc...,True,0.01,5,1.499768e+06,74282.8660,2026-07-31,2025-05-02,True,12123.350347,77363.410015,129505.415150,1.499768e+06,"[""85014971590839487133161357681037732937544902...",500,5,12123.350347,77363.410015,129505.415150,1.499768e+06,1.499768e+06,74282.8660,0.0,True,False,,"[{'id': '23784', 'ticker': 'what-will-happen-b...",False,False,2025-05-02T15:47:37Z,False,0.997979,False,True,"[{'id': '22695', 'conditionId': '0x9c1a953fe92...",20,3.5,0.01,0.010,0.01,0.54,0.54,0.55,True,True,,False,False,False,False,[],False,False,2025-05-02T15:47:04.719613Z,False,False,False,False,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,540817,New Rihanna Album before GTA VI?,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1...,new-rhianna-album-before-gta-vi-926,,2026-07-31T12:00:00Z,51898.1321,2025-05-02T15:48:10.582Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if Rihanna o...","[""Yes"", ""No""]","[""0.575"", ""0.425""]",693077.5510450028,True,False,,2025-05-02T15:04:43.762151Z,2026-04-13T18:48:13.58067Z,False,False,0x91430CaD2d3975766499717fA0D66A78D814E5c5,False,0x6A9D222616C90FcA5754cd1333cFD9b7fb6a4F74,True,New Rihanna Album,2,0xb6fd5ea8c21f01471ad673950edd4a1645698946906a...,True,0.01,5,6.930776e+05,51898.1321,2026-07-31,2025-05-02,True,1488.407778,8951.111746,38981.782767,6.930776e+05,"[""98022490269692409998126496127597032490334070...",500,5,1488.407778,8951.111746,38981.782767,6.930776e+05,6.930776e+05,51898.1321,0.0,True,False,,"[{'id': '23784', 'ticker': 'what-will-happen-b...",False,False,2025-05-02T15:47:45Z,False,0.994406,False,True,"[{'id': '22694', 'conditionId': '0x1fad72fae20...",20,3.5,0.01,0.010,0.05,0.57,0.57,0.58,True,True,,False,False,False,False,[],False,False,2025-05-02T15:47:04.727292Z,False,False,False,False,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,540818,New Playboi Carti Album before GTA VI?,0x50ddb9cd80d5c271664a2ebb7fcaed1d0a148d82c8e8...,new-playboi-carti-album-before-gta-vi-421,,2026-07-31T12:00:00Z,47237.6203,2025-05-02T15:48:10.837Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east

,id,ticker,slug,title,description,resolutionSource,startDate,creationDate,endDate,image,icon,active,closed,archived,new,featured,restricted,liquidity,volume,openInterest,createdAt,updatedAt,competitive,volume24hr,volume1wk,volume1mo,volume1yr,enableOrderBook,liquidityClob,negRisk,commentCount,markets,tags,cyom,showAllOutcomes,showMarketImages,enableNegRisk,automaticallyActive,gmpChartMode,negRiskAugmented,cumulativeMarkets,pendingDeployment,deploying,requiresTranslation,estimateValue,eventMetadata,startTime,countryName,sortBy,eventCreators,negRiskMarketID,deployingTimestamp,series,seriesSlug,eventDate,color,featuredOrder,createdBy,electionType,eventWeek,score,elapsed,period,live,ended,finishedTimestamp,gameId,teams,cantEstimate,estimatedValue,parentEventId,tweetCount,liquidityAmm,derived_category
0,16167,microstrategy-sell-any-bitcoin-in-2025,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",,2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,2025-12-31T12:00:00Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,True,False,False,False,False,True,62388.62032,2.216768e+07,559904.935146,2024-12-31T16:02:31.965903Z,2026-04-13T18:52:41.883678Z,0.870909,18792.222272,9.162357e+06,1.582935e+07,1.944539e+07,True,62388.62032,False,233,"[{'id': '516926', 'question': 'MicroStrategy s...","[{'id': '120', 'label': 'Finance', 'slug': 'fi...",False,True,False,False,True,default,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Finance
1,16183,kraken-ipo-in-2025,kraken-ipo-in-2025,Kraken IPO by ___ ?,"This market will resolve to ""Yes"" if Kraken (U...",,2024-12-31T19:05:45.901363Z,2024-12-31T19:05:45.901361Z,2025-12-31T12:00:00Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,True,False,False,False,False,True,8410.15680,1.474720e+06,46690.538576,2024-12-31T18:18:36.692417Z,2026-04-13T18:52:53.938814Z,0.990099,11340.277452,8.729781e+04,2.903828e+05,9.264497e+05,True,8410.15680,False,43,"[{'id': '516950', 'question': 'Kraken IPO in 2...","[{'id': '101300', 'label': 'exchange', 'slug':...",False,True,False,False,True,default,False,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,exchange
2,16263,macron-out-in-2025,macron-out-in-2025,Macron out by...?,"This market will resolve to ""Yes"" if Emmanuel ...",,2025-01-03T19:35:04.095066Z,2025-01-03T19:35:04.095064Z,2026-06-30T12:00:00Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,True,False,False,False,False,True,88377.13693,1.902676e+06,73146.032072,2025-01-03T19:28:39.855536Z,2026-04-13T18:52:44.662094Z,0.816226,1237.218978,5.119185e+04,4.530689e+05,1.902676e+06,True,88377.13693,False,91,"[{'id': '517231', 'question': 'Macron out in 2...","[{'id': '1378', 'label': 'France', 'slug': 'fr...",False,True,False,False,True,default,False,False,False,False,False,NaN,{'context_description': 'French President Emma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,France


Top derived categories in open events:


,count
derived_category,
Sports,3895
Up or Down,2247
Crypto,517
Politics,512
Esports,328
Crypto Prices,247
Elections,226
Weather,146
Tennis,127


## 4. Clean and Normalize Market Data


In [4]:
def normalize_markets(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    flat = pd.json_normalize(df.to_dict(orient="records"), sep=".")

    for c in ["startDate", "endDate", "createdAt", "updatedAt", "closedTime", "acceptingOrdersTimestamp"]:
        if c in flat.columns:
            flat[c] = pd.to_datetime(flat[c], errors="coerce", utc=True)

    for c in ["volume", "liquidity", "volumeNum", "liquidityNum", "lastTradePrice", "bestBid", "bestAsk"]:
        if c in flat.columns:
            flat[c] = pd.to_numeric(flat[c], errors="coerce")

    if "active" in flat.columns:
        flat["active"] = flat["active"].fillna(False)
    if "closed" in flat.columns:
        flat["closed"] = flat["closed"].fillna(False)

    return flat


markets = normalize_markets(markets_raw)
events = pd.json_normalize(events_raw.to_dict(orient="records"), sep=".") if not events_raw.empty else events_raw

print("Normalized markets shape:", markets.shape)
print("Normalized events shape:", events.shape)


Normalized markets shape: (2000, 93)
Normalized events shape: (10778, 84)


## 5. Analyze Market Prices, Volume, and Activity


In [5]:
if markets.empty:
    print("No market data available to analyze.")
else:
    summary = {
        "market_count": len(markets),
        "active_count": int(markets["active"].sum()) if "active" in markets.columns else np.nan,
        "closed_count": int(markets["closed"].sum()) if "closed" in markets.columns else np.nan,
        "total_volume": float(markets["volume"].sum()) if "volume" in markets.columns else np.nan,
        "total_liquidity": float(markets["liquidity"].sum()) if "liquidity" in markets.columns else np.nan,
    }
    print(pd.Series(summary))

    cols_for_view = [c for c in ["question", "slug", "active", "closed", "volume", "liquidity", "lastTradePrice"] if c in markets.columns]

    if "volume" in markets.columns:
        print("\nTop markets by volume")
        display(markets.sort_values("volume", ascending=False)[cols_for_view].head(10))

    if "liquidity" in markets.columns:
        print("\nTop markets by liquidity")
        display(markets.sort_values("liquidity", ascending=False)[cols_for_view].head(10))


market_count       2.000000e+03
active_count       2.000000e+03
closed_count       0.000000e+00
total_volume       3.734267e+09
total_liquidity    2.838505e+08
dtype: float64

Top markets by volume


,question,slug,active,closed,volume,liquidity,lastTradePrice
174,Will LeBron James win the 2028 US Presidential...,will-lebron-james-win-the-2028-us-presidential...,True,False,4.641779e+07,1.848744e+06,0.005
139,Will Chelsea Clinton win the 2028 Democratic p...,will-chelsea-clinton-win-the-2028-democratic-p...,True,False,4.571992e+07,7.638164e+05,0.008
142,Will Oprah Winfrey win the 2028 Democratic pre...,will-oprah-winfrey-win-the-2028-democratic-pre...,True,False,4.453247e+07,7.600069e+05,0.010
143,Will Andrew Yang win the 2028 Democratic presi...,will-andrew-yang-win-the-2028-democratic-presi...,True,False,4.376878e+07,1.177300e+06,0.007
134,Will Bernie Sanders win the 2028 Democratic pr...,will-bernie-sanders-win-the-2028-democratic-pr...,True,False,4.145225e+07,2.412071e+06,0.006
136,Will LeBron James win the 2028 Democratic pres...,will-lebron-james-win-the-2028-democratic-pres...,True,False,3.971257e+07,2.208033e+06,0.007
170,Will Tim Walz win the 2028 US Presidential Ele...,will-tim-walz-win-the-2028-us-presidential-ele...,True,False,3.957396e+07,1.703467e+06,0.006
132,Will Hillary Clinton win the 2028 Democratic p...,will-hillary-clinton-win-the-2028-democratic-p...,True,False,3.890392e+07,1.960728e+06,0.007
138,Will George Clooney win the 2028 Democratic pr...,will-george-clooney-win-the-2028-democratic-pr...,True,False,3.841612e+07,1.564783e+06,0.007
121,Will Tim Walz win the 2028 Democratic presiden...,will-tim-walz-win-the-2028-democratic-presiden...,True,False,3.804537e+07,1.261216e+06,0.008



Top markets by liquidity


,question,slug,active,closed,volume,liquidity,lastTradePrice
99,Will Panama win the 2026 FIFA World Cup?,will-team-z-win-the-2026-fifa-world-cup-316,True,False,3.528239e+06,4.123785e+06,0.002
102,Will Iraq win the 2026 FIFA World Cup?,will-iraq-win-the-2026-fifa-world-cup,True,False,3.994898e+06,3.990573e+06,0.001
97,Will Haiti win the 2026 FIFA World Cup?,will-haiti-win-the-2026-fifa-world-cup,True,False,1.494941e+07,3.988784e+06,0.001
103,Will Bosnia-Herzegovina win the 2026 FIFA Worl...,will-bosnia-herzegovina-win-the-2026-fifa-worl...,True,False,3.652577e+06,3.957486e+06,0.004
104,Will Cezchia win the 2026 FIFA World Cup?,will-cezchia-win-the-2026-fifa-world-cup,True,False,2.310059e+06,3.871085e+06,0.003
87,Will Ghana win the 2026 FIFA World Cup?,will-ghana-win-the-2026-fifa-world-cup,True,False,1.326829e+07,3.802073e+06,0.003
82,Will Jordan win the 2026 FIFA World Cup?,will-jordan-win-the-2026-fifa-world-cup-233,True,False,1.894530e+07,3.758532e+06,0.001
78,Will Australia win the 2026 FIFA World Cup?,will-australia-win-the-2026-fifa-world-cup-816,True,False,1.217510e+07,3.708312e+06,0.002
74,Will Tunisia win the 2026 FIFA World Cup?,will-tunisia-win-the-2026-fifa-world-cup-165,True,False,1.384948e+07,3.587509e+06,0.002
90,Will Cape Verde win the 2026 FIFA World Cup?,will-cape-verde-win-the-2026-fifa-world-cup,True,False,1.448491e+07,3.574379e+06,0.003


## 6. Visualize Key Event Trends


In [ ]:
if events_raw.empty:
    print("No event data available to visualize.")
else:
    events_viz = events_raw.copy()

    volume_col = None
    for c in ["volume", "volumeNum", "volumeClob", "liquidity", "openInterest"]:
        if c in events_viz.columns:
            volume_col = c
            break

    name_col = None
    for c in ["title", "slug", "ticker"]:
        if c in events_viz.columns:
            name_col = c
            break

    if volume_col is None:
        print("No volume-like column found in events data.")
    else:
        events_viz[volume_col] = pd.to_numeric(events_viz[volume_col], errors="coerce").fillna(0)

        # 1) Event volume distribution
        plt.figure(figsize=(11, 4))
        sns.histplot(events_viz[volume_col], bins=40)
        plt.title(f"Open Event {volume_col} Distribution")
        plt.xlabel(volume_col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()

        # 2) Top events by volume-like metric
        if name_col is not None:
            top_events = events_viz.sort_values(volume_col, ascending=False).head(15).copy()
            top_events[name_col] = top_events[name_col].astype(str)

            plt.figure(figsize=(12, 6))
            sns.barplot(data=top_events, y=name_col, x=volume_col)
            plt.title(f"Top 15 Open Events by {volume_col}")
            plt.xlabel(volume_col)
            plt.ylabel("Event")
            plt.tight_layout()
            plt.show()

    # 3) Open event creation trend
    date_col = None
    for c in ["createdAt", "creationDate", "startDate", "startTime"]:
        if c in events_viz.columns:
            date_col = c
            break

    if date_col is not None:
        events_viz[date_col] = pd.to_datetime(events_viz[date_col], errors="coerce", utc=True)
        created_daily = (
            events_viz.dropna(subset=[date_col])
            .assign(day=lambda d: d[date_col].dt.date)
            .groupby("day", as_index=False)
            .size()
            .rename(columns={"size": "open_events_count"})
        )

        if not created_daily.empty:
            plt.figure(figsize=(12, 4))
            sns.lineplot(data=created_daily, x="day", y="open_events_count")
            plt.title("Open Events Over Time")
            plt.xlabel("Date")
            plt.ylabel("Open Event Count")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

    # 4) Derived category concentration (if available)
    if "derived_category" in events_viz.columns:
        cat_counts = (
            events_viz["derived_category"]
            .fillna("uncategorized")
            .astype(str)
            .value_counts()
            .head(12)
            .reset_index()
        )
        cat_counts.columns = ["derived_category", "event_count"]

        plt.figure(figsize=(12, 5))
        sns.barplot(data=cat_counts, x="derived_category", y="event_count")
        plt.title("Top Derived Categories (Open Events)")
        plt.xlabel("Derived Category")
        plt.ylabel("Event Count")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()


## 7. Save Processed Results to Files


In [8]:
import os

output_dir = "./polymarket_outputs"
os.makedirs(output_dir, exist_ok=True)

markets_path = os.path.join(output_dir, "markets_cleaned.csv")
events_path = os.path.join(output_dir, "events_open_raw.csv")

# to_csv writes with mode='w' by default (overwrite existing files).
if not markets.empty:
    markets.to_csv(markets_path, index=False, mode="w")

if not events_raw.empty:
    events_raw.to_csv(events_path, index=False, mode="w")

print(f"Saved files in: {os.path.abspath(output_dir)}")
if os.path.exists(markets_path):
    print("-", markets_path)
if os.path.exists(events_path):
    print("-", events_path)


Saved files in: /Users/qisongqiao/Warehouse/nyu_course_homework/TECH-GB.2148-01_DealingwithData/project/polymarket_outputs
- ./polymarket_outputs/markets_cleaned.csv
- ./polymarket_outputs/events_open_raw.csv


## 8. Top 5 Open Events by Volume

Display the top 5 open events by volume so you can choose one event ID for section 9.


In [20]:
if events_raw.empty:
    raise ValueError("events_raw is empty. Run Section 2 first.")

events_open = events_raw.copy()

volume_col = next((c for c in ["volume", "volumeNum", "volumeClob", "liquidity"] if c in events_open.columns), None)
if volume_col is None:
    raise ValueError("No volume-like column found in events data.")

events_open[volume_col] = pd.to_numeric(events_open[volume_col], errors="coerce").fillna(0)

top_events = events_open.sort_values(volume_col, ascending=False).head(5).copy()
if top_events.empty:
    raise ValueError("No open events found after sorting by volume.")

show_cols = [c for c in ["id", "title", "slug", volume_col, "active", "closed", "derived_category"] if c in top_events.columns]
print(f"Top {len(top_events)} open events by {volume_col}:")
display(top_events[show_cols])

print("\nCopy one event id from the table above and set SELECTED_EVENT_ID in section 9.")


Top 5 open events by volume:


,id,title,slug,volume,active,closed,derived_category
33,30829,Democratic Presidential Nominee 2028,democratic-presidential-nominee-2028,1.037398e+09,True,False,World Elections
30,30615,2026 FIFA World Cup Winner,2026-fifa-world-cup-winner-595,6.240525e+08,True,False,Soccer
37,31875,Republican Presidential Nominee 2028,republican-presidential-nominee-2028,5.496272e+08,True,False,Politics
35,31552,Presidential Election Winner 2028,presidential-election-winner-2028,5.212263e+08,True,False,World Elections
49,33507,English Premier League Winner,english-premier-league-winner,3.163739e+08,True,False,Sports



Copy one event id from the table above and set SELECTED_EVENT_ID in section 9.


## 9. Select One Event ID -> Vertex AI Suggestion

Set `SELECTED_EVENT_ID` from section 8 output, then ask Vertex AI for suggestions only for that event.


In [24]:
import os

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv()

if events_raw.empty:
    raise ValueError("events_raw is empty. Run Section 2 first.")

# Rebuild top_events if section 8 has not been run.
if "top_events" not in globals() or top_events is None or len(top_events) == 0:
    events_open = events_raw.copy()
    volume_col = next((c for c in ["volume", "volumeNum", "volumeClob", "liquidity"] if c in events_open.columns), None)
    if volume_col is None:
        raise ValueError("No volume-like column found in events data.")
    events_open[volume_col] = pd.to_numeric(events_open[volume_col], errors="coerce").fillna(0)
    top_events = events_open.sort_values(volume_col, ascending=False).head(5).copy()
else:
    volume_col = next((c for c in ["volume", "volumeNum", "volumeClob", "liquidity"] if c in top_events.columns), None)
    if volume_col is None:
        volume_col = next((c for c in ["volume", "volumeNum", "volumeClob", "liquidity"] if c in events_raw.columns), None)

SELECTED_EVENT_ID = "30615"

if not str(SELECTED_EVENT_ID).strip():
    raise ValueError("Please set SELECTED_EVENT_ID in this cell using one id from section 8 output.")

selected = top_events[top_events["id"].astype(str) == str(SELECTED_EVENT_ID).strip()].copy()
if selected.empty:
    available = ", ".join(top_events["id"].astype(str).tolist())
    raise ValueError(f"Selected id {SELECTED_EVENT_ID!r} not in top 5 list. Available ids: {available}")

row = selected.iloc[0]
show_cols = [c for c in ["id", "title", "slug", volume_col, "derived_category"] if c in selected.columns]
print("Selected event:")
display(selected[show_cols])

event_title = str(row.get("title", row.get("question", row.get("slug", ""))))
event_slug = str(row.get("slug", ""))
event_volume = float(row.get(volume_col, 0) or 0)
event_desc = str(row.get("description", ""))[:1200]

a_prompt = (
    "You are a prediction-market trading analyst. "
    "Given this one open Polymarket event, provide concise trading suggestions. "
    "Include: thesis, entry idea, key risk, and invalidation condition.\n\n"
    f"event_id: {row.get('id', '')}\n"
    f"event_title: {event_title}\n"
    f"event_slug: {event_slug}\n"
    f"{volume_col}: {event_volume:,.2f}\n"
    f"description: {event_desc}"
)

VERTEX_PROJECT_ID = os.getenv("VERTEX_PROJECT_ID", "").strip()
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1").strip() or "us-central1"
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash").strip() or "gemini-2.5-flash"
VERTEX_ACCESS_TOKEN = os.getenv("VERTEX_ACCESS_TOKEN", "").strip()

if not VERTEX_PROJECT_ID or not VERTEX_ACCESS_TOKEN:
    print("Vertex credentials missing. Set VERTEX_PROJECT_ID and VERTEX_ACCESS_TOKEN in .env")
    print("Prompt:
")
    print(a_prompt)
else:
    endpoint = (
        f"https://{VERTEX_LOCATION}-aiplatform.googleapis.com/v1/projects/{VERTEX_PROJECT_ID}/locations/{VERTEX_LOCATION}"
        f"/publishers/google/models/{VERTEX_MODEL}:generateContent"
    )
    headers = {
        "Authorization": f"Bearer {VERTEX_ACCESS_TOKEN}",
        "Content-Type": "application/json",
    }

    def call_vertex(prompt_text: str) -> tuple[str, str]:
        payload = {
            "contents": [{"role": "user", "parts": [{"text": prompt_text}]}],
            "generationConfig": {"temperature": 0.3, "maxOutputTokens": 2048},
        }
        r = requests.post(endpoint, headers=headers, json=payload, timeout=60)
        r.raise_for_status()
        out = r.json()

        candidates = out.get("candidates", []) if isinstance(out, dict) else []
        if not candidates:
            return "", ""

        c0 = candidates[0] if isinstance(candidates[0], dict) else {}
        finish_reason = str(c0.get("finishReason", "")).strip()
        parts = c0.get("content", {}).get("parts", [])
        text = "".join(
            str(part.get("text", ""))
            for part in parts
            if isinstance(part, dict) and part.get("text") is not None
        ).strip()
        return text, finish_reason

    try:
        chunks = []
        text, finish = call_vertex(a_prompt)
        if text:
            chunks.append(text)

        for _ in range(3):
            if finish != "MAX_TOKENS":
                break
            cont_prompt = (
                "Continue exactly where you left off. Do not repeat prior text.\n\n"
                f"Original task:\n{a_prompt}\n\n"
                f"Already generated (tail):\n{''.join(chunks)[-2000:]}"
            )
            text, finish = call_vertex(cont_prompt)
            if not text:
                break
            chunks.append(text)

        suggestions = "\n".join([x for x in chunks if x]).strip()
        print("AI trading suggestions:\n")
        print(suggestions if suggestions else "(empty model response)")
    except Exception:
        print("Vertex call failed. Prompt:\n")
        print(a_prompt)


Selected event:


,id,title,slug,volume,derived_category
30,30615,2026 FIFA World Cup Winner,2026-fifa-world-cup-winner-595,6.240525e+08,Soccer


AI trading suggestions:

Here's a concise trading suggestion for the 2026 FIFA World Cup Winner market:

**Team Focus: Germany**

*   **Thesis:** Germany, a perennial football powerhouse, is currently undervalued due to recent underperformance. With Euro 2024 on home soil providing a potential resurgence and a strong pipeline of young talent, they have significant upside to be a top contender by 2026. The market likely overweights their recent struggles and underweights their potential for a strong rebuild under a new coaching setup.
*   **Entry Idea:** Buy "Yes" on Germany if their price is below 10-12%. This price range offers good value for a team with their historical pedigree and potential for a strong comeback. (Note: Assumes current market prices reflect recent performance).
*   **Key Risk:** Failure to integrate new talent effectively, continued tactical issues, or a poor showing at Euro 2024 which could further depress their market value and confidence.
*   **Invalidation Cond